In [1]:
"""
GRACE — Graph-Aware Complexity Estimation
==========================================
A self-contained, verified Python implementation of the GRACE algorithm
for DAG-based query complexity estimation and agentic task routing.

Run as: python GRACE_notebook.py
Or paste each cell block into a Jupyter notebook.
"""

# ──────────────────────────────────────────────────────────────────────────────
# CELL 1 — Imports & Configuration
# ──────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import json
import math
import os
import re
import statistics
import textwrap
from dataclasses import dataclass, field
from enum import Enum
from typing import Dict, List, Optional, Tuple

import networkx as nx
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


/Users/poorna/Downloads/research/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from __future__ import annotations
 
import json
import logging
import os
import re
from typing import Any
from groq import Groq
from dotenv import load_dotenv
load_dotenv()
from pathlib import Path
from typing import Dict
import pandas as pd
from pydantic import BaseModel, Field, field_validator, model_validator
logger = logging.getLogger(__name__)

In [4]:
# ── Anthropic client (set ANTHROPIC_API_KEY in env or leave as None for demo)
try:
    from groq import Groq
    _CLIENT = Groq(api_key=api_key or os.environ.get("GROQ_API_KEY"))
    _USE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))
except Exception:
    _CLIENT = None
    _USE_LLM = False

# ── Sentence encoder (all-MiniLM-L6-v2 is small, fast, good for cosine sim)
_ENCODER = SentenceTransformer("all-MiniLM-L6-v2")

print("✓ Imports OK | LLM calls:", "ENABLED" if _USE_LLM else "DEMO MODE")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 25501.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Imports OK | LLM calls: DEMO MODE


In [6]:


class Domain(str, Enum):
    """Supported task domains with pre-calibrated weight vectors."""
    GENERAL       = "general"
    CODE          = "code"
    RESEARCH      = "research"
    FINANCE       = "finance"
    MATH          = "math"
    MULTIMODAL    = "multimodal"
    AIASSITATANT  = "ai_assistant"


# class RoutingTier(str, Enum):
#     """Four routing tiers in ascending complexity order."""
#     LLM_CALL      = "LLM Call"
#     REASONING     = "Reasoning Workflow"
#     SINGLE_AGENT  = "Single Agent"
#     MULTI_AGENT   = "Multi-Agent"


In [8]:

# ───────────────

# ──────────────────────────────────────────────────────────────────────────────
# CELL 3 — Data Structures
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class SubTask:
    """Represents one node in the task DAG."""
    id: int
    description: str
    output_description: str = ""    # what this subtask produces
    input_requirement: str  = ""    # what this subtask needs as input

    # Per-node signals (all ∈ [0, 1])
    R: float = 0.0   # Reasoning depth
    X: float = 0.0   # External tool requirement
    M: float = 0.0   # State / memory dependency
    U: float = 0.0   # Node uncertainty (entropy from decomposition)


@dataclass
class GRACEResult:
    """Full output of one GRACE estimation."""
    query: str
    subtasks: List[SubTask]
    dag: nx.DiGraph

    # Structural signals
    critical_path_len: int   = 0
    parallelism_ratio: float = 0.0
    mean_in_degree: float    = 0.0
    mean_edge_coupling: float = 0.0

    # Aggregated node signals
    R_bar: float = 0.0
    X_bar: float = 0.0
    M_bar: float = 0.0
    U_bar: float = 0.0

    # Output
    complexity_score: float  = 0.0
    uncertainty: float       = 0.0
    # routing_tier: RoutingTier = RoutingTier.LLM_CALL
    domain: Domain            = Domain.GENERAL
    weights: Dict[str, float] = field(default_factory=dict)

    def summary(self) -> str:
        lines = [
            f"\n{'═'*60}",
            f"  GRACE Complexity Estimation",
            f"{'═'*60}",
            f"  Query      : {textwrap.shorten(self.query, 60)}",
            f"  Domain     : {self.domain.value}",
            f"  Subtasks   : {len(self.subtasks)}",
            f"  Crit Path  : {self.critical_path_len}",
            f"  Parallelism: {self.parallelism_ratio:.2f}",
            f"  Mean κ     : {self.mean_edge_coupling:.3f}",
            f"  R̄={self.R_bar:.2f}  X̄={self.X_bar:.2f}  "
            f"M̄={self.M_bar:.2f}  Ū={self.U_bar:.2f}",
            f"{'─'*60}",
            f"  Complexity C  : {self.complexity_score:.4f}",
            f"  Uncertainty σ : {self.uncertainty:.4f}",
            # f"  ➜ Routing Tier : {self.routing_tier.value}",
            f"{'═'*60}\n",
        ]
        return "\n".join(lines)


In [10]:

# ──────────────────────────────────────────────────────────────────────────────
# CELL 4 — Domain Weight Registry
# ──────────────────────────────────────────────────────────────────────────────

# Weight vector keys: (wl=critical_path, wx=tool, wm=memory, ws=coupling, wk=indegree)
# Each vector sums to 1.0 and is hand-calibrated; replace with a trained
# linear classifier for production use (see Cell 10).

DOMAIN_WEIGHTS: Dict[Domain, Dict[str, float]] = {
    Domain.GENERAL    : {"wl": 0.25, "wx": 0.20, "wm": 0.20, "ws": 0.20, "wk": 0.15},
    Domain.CODE       : {"wl": 0.20, "wx": 0.35, "wm": 0.15, "ws": 0.15, "wk": 0.15},
    Domain.RESEARCH   : {"wl": 0.30, "wx": 0.25, "wm": 0.15, "ws": 0.20, "wk": 0.10},
    Domain.FINANCE    : {"wl": 0.20, "wx": 0.25, "wm": 0.25, "ws": 0.20, "wk": 0.10},
    Domain.MATH       : {"wl": 0.35, "wx": 0.15, "wm": 0.10, "ws": 0.25, "wk": 0.15},
    Domain.MULTIMODAL : {"wl": 0.20, "wx": 0.30, "wm": 0.20, "ws": 0.15, "wk": 0.15},
    Domain.AIASSITATANT : {"wl": 0.20, "wx": 0.30, "wm": 0.20, "ws": 0.15, "wk": 0.15}
}

def _validate_weights() -> None:
    """Assert every weight vector sums to 1.0 ± ε."""
    for domain, w in DOMAIN_WEIGHTS.items():
        total = sum(w.values())
        assert abs(total - 1.0) < 1e-9, \
            f"Weight vector for {domain} sums to {total}, expected 1.0"
    print("✓ All domain weight vectors validated (sum = 1.0)")

_validate_weights()


✓ All domain weight vectors validated (sum = 1.0)


In [11]:
# ──────────────────────────────────────────────────────────────────────────────
# CELL 5 — Routing Thresholds
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class RoutingThresholds:
    """
    θ₁, θ₂, θ₃ separate the four tiers.
    δ_low / δ_high gate uncertainty-based escalation.
    Calibrate on a held-out validation set per deployment domain.
    """
    theta_1: float = 0.25   # LLM Call  → Reasoning Workflow
    theta_2: float = 0.50   # Reasoning → Single Agent
    theta_3: float = 0.75   # Single    → Multi-Agent
    delta_low: float  = 0.30
    delta_high: float = 0.60

DEFAULT_THRESHOLDS = RoutingThresholds()

In [15]:


_DECOMPOSE_SYSTEM = textwrap.dedent("""
    You are a task decomposition engine. Given a user query, output ONLY valid JSON.
    Decompose the query into 2-6 atomic subtasks and their dependencies.

    Output schema (strict JSON, no markdown):
    {
      "subtasks": [
        {
          "id": 0,
          "description": "...",
          "output_description": "what this subtask produces",
          "input_requirement": "what this subtask needs as input",
          "depends_on": []          // list of integer ids this subtask depends on
        }
      ]
    }

    Rules:
    - id values must be unique integers starting at 0
    - depends_on must only reference ids that appear earlier in the list
    - The graph formed by depends_on must be acyclic (a valid DAG)
""").strip()


In [16]:

def _llm_decompose(query: str) -> dict:
    """Call Claude to decompose the query. Returns raw parsed JSON."""
    response = _CLIENT.messages.create(
        model="moonshotai/kimi-k2-instruct",
        max_tokens=1024,
        system=_DECOMPOSE_SYSTEM,
        messages=[{"role": "user", "content": query}],
    )
    raw = response.content[0].text.strip()
    # Strip accidental markdown fences
    raw = re.sub(r"^```(?:json)?\n?", "", raw)
    raw = re.sub(r"\n?```$", "", raw)
    return json.loads(raw)

In [17]:



def decompose_and_build_dag(query: str) -> Tuple[List[SubTask], nx.DiGraph]:
    """
    Steps 1 & 2: Decompose query into subtasks and build the dependency DAG.
    Returns (subtask_list, directed_acyclic_graph).
    """
    raw = _llm_decompose(query) if _USE_LLM else _demo_decompose(query)

    subtasks: List[SubTask] = []
    id_map: Dict[int, SubTask] = {}

    for item in raw["subtasks"]:
        st = SubTask(
            id=item["id"],
            description=item["description"],
            output_description=item.get("output_description", ""),
            input_requirement=item.get("input_requirement", ""),
        )
        subtasks.append(st)
        id_map[st.id] = st

    G = nx.DiGraph()
    for item in raw["subtasks"]:
        G.add_node(item["id"], subtask=id_map[item["id"]])
        for dep in item.get("depends_on", []):
            G.add_edge(dep, item["id"])   # dep → item (dep must complete first)

    # Integrity check
    assert nx.is_directed_acyclic_graph(G), \
        "Decomposition produced a cyclic graph — rejecting."

    return subtasks, G

In [18]:


# ──────────────────────────────────────────────────────────────────────────────
# CELL 7 — Step 3: Per-Node Signal Scoring
# ──────────────────────────────────────────────────────────────────────────────

# Keyword heuristics for fast, deterministic signal estimation.
# Replace individual methods with a small classifier for higher accuracy.

_REASONING_HIGH = {"plan", "synthesize", "design", "create", "strategize",
                   "infer", "reason", "evaluate", "judge", "critique"}
_REASONING_MED  = {"explain", "compare", "summarize", "analyse", "analyze",
                   "infer", "multi-step", "deduce"}
_TOOL_HIGH      = {"execute", "run", "deploy", "fetch", "api", "database",
                   "query", "scrape", "download", "upload", "compute", "test"}
_TOOL_MED       = {"search", "retrieve", "look up", "call", "read file",
                   "write file", "send"}
_MEMORY_HIGH    = {"persistent", "across turns", "remember", "history",
                   "session", "stateful", "maintain state", "track"}
_MEMORY_MED     = {"context", "previous", "based on earlier", "follow-up",
                   "given above"}


def _keyword_score(text: str, high_set: set, med_set: set) -> float:
    """Return 0, 0.5, or 1.0 based on keyword presence."""
    t = text.lower()
    if any(k in t for k in high_set):
        return 1.0
    if any(k in t for k in med_set):
        return 0.5
    return 0.0


In [ ]:






def _uncertainty_from_text(text: str) -> float:
    """
    Proxy for decomposition-step token entropy.
    Uses description length & hedging-word count as a cheap heuristic.
    Replace with actual token-level entropy from the LLM's logprobs in prod.
    """
    hedge_words = {"maybe", "possibly", "might", "could", "unclear",
                   "uncertain", "depending", "if", "or", "either"}
    words = text.lower().split()
    if not words:
        return 0.0
    hedge_ratio = sum(1 for w in words if w in hedge_words) / len(words)
    # Longer descriptions with more hedging = higher uncertainty
    length_signal = min(len(words) / 30, 1.0)   # normalise at 30 words
    return float(np.clip(0.5 * hedge_ratio * 10 + 0.5 * length_signal, 0, 1))


def score_nodes(subtasks: List[SubTask]) -> None:
    """Step 3 — Compute R, X, M, U for every subtask in-place."""
    for st in subtasks:
        combined = f"{st.description} {st.output_description} {st.input_requirement}"
        st.R = _keyword_score(combined, _REASONING_HIGH, _REASONING_MED)
        st.X = _keyword_score(combined, _TOOL_HIGH, _TOOL_MED)
        st.M = _keyword_score(combined, _MEMORY_HIGH, _MEMORY_MED)
        st.U = _uncertainty_from_text(combined)


# ──────────────────────────────────────────────────────────────────────────────
# CELL 8 — Step 4: Per-Edge Semantic Coupling κ
# ──────────────────────────────────────────────────────────────────────────────

def compute_edge_coupling(
    subtasks: List[SubTask],
    G: nx.DiGraph,
) -> Dict[Tuple[int, int], float]:
    """
    Step 4 — κ(i,j) = cosine similarity between tᵢ's output embedding
    and tⱼ's input embedding. High κ → tight semantic chaining.
    Returns a dict keyed by (source_id, target_id).
    """
    id_map = {st.id: st for st in subtasks}
    kappas: Dict[Tuple[int, int], float] = {}

    if not G.edges():
        return kappas

    # Batch-encode all output/input descriptions in one pass (fast)
    edge_list = list(G.edges())
    src_texts = [id_map[s].output_description or id_map[s].description
                 for s, _ in edge_list]
    tgt_texts = [id_map[t].input_requirement or id_map[t].description
                 for _, t in edge_list]

    src_embs = _ENCODER.encode(src_texts, convert_to_numpy=True)
    tgt_embs = _ENCODER.encode(tgt_texts, convert_to_numpy=True)

    for idx, (s, t) in enumerate(edge_list):
        sim = float(cosine_similarity(
            src_embs[idx].reshape(1, -1),
            tgt_embs[idx].reshape(1, -1)
        )[0][0])
        kappas[(s, t)] = float(np.clip(sim, 0.0, 1.0))

    return kappas


# ──────────────────────────────────────────────────────────────────────────────
# CELL 9 — Step 5: Graph-Structural Features
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class GraphFeatures:
    critical_path_len: int    # L  — number of nodes on the longest path
    parallelism_ratio: float  # P  — max parallel width / |N|
    mean_in_degree: float     # K  — average in-degree of non-source nodes
    mean_edge_coupling: float # K̄  — mean κ across all edges


def extract_graph_features(
    G: nx.DiGraph,
    kappas: Dict[Tuple[int, int], float],
) -> GraphFeatures:
    """Step 5 — Compute L, P, K, K̄ from the DAG topology."""
    n = G.number_of_nodes()
    assert n > 0, "DAG has no nodes"

    # L — critical path length (node count on longest path)
    cp = nx.dag_longest_path(G)
    L = len(cp)

    # P — parallelism ratio: widest generation / total nodes
    generations = list(nx.topological_generations(G))
    max_width = max(len(gen) for gen in generations)
    P = max_width / n if n > 0 else 0.0

    # K — mean in-degree of non-source nodes
    non_sources = [v for v, d in G.in_degree() if d > 0]
    K = (sum(G.in_degree(v) for v in non_sources) / len(non_sources)
         if non_sources else 0.0)

    # K̄ — mean edge coupling
    K_bar = float(np.mean(list(kappas.values()))) if kappas else 0.0

    return GraphFeatures(
        critical_path_len=L,
        parallelism_ratio=P,
        mean_in_degree=K,
        mean_edge_coupling=K_bar,
    )


# ──────────────────────────────────────────────────────────────────────────────
# CELL 10 — Step 6 & 7: Aggregate Signals + Domain-Adaptive Weights
# ──────────────────────────────────────────────────────────────────────────────

def aggregate_node_signals(
    subtasks: List[SubTask],
) -> Tuple[float, float, float, float]:
    """Step 6 — Return (R̄, X̄, M̄, Ū) as arithmetic means."""
    R_bar = float(np.mean([st.R for st in subtasks]))
    X_bar = float(np.mean([st.X for st in subtasks]))
    M_bar = float(np.mean([st.M for st in subtasks]))
    U_bar = float(np.mean([st.U for st in subtasks]))
    return R_bar, X_bar, M_bar, U_bar


_DOMAIN_KEYWORDS: Dict[Domain, List[str]] = {
    Domain.CODE       : ["code", "implement", "function", "debug", "script",
                         "program", "algorithm", "class", "api"],
    Domain.RESEARCH   : ["research", "literature", "survey", "paper", "study",
                         "find", "search", "analyse", "analyze"],
    Domain.FINANCE    : ["stock", "market", "invest", "financial", "portfolio",
                         "price", "revenue", "trading", "valuation"],
    Domain.MATH       : ["calculate", "solve", "equation", "proof", "derivative",
                         "integral", "matrix", "probability", "math"],
    Domain.MULTIMODAL : ["image", "audio", "video", "visual", "photo",
                         "chart", "diagram", "ocr", "transcribe"],
}


def detect_domain(query: str) -> Domain:
    """
    Step 7 — Lightweight rules-based domain detector.
    Returns the domain with the most keyword hits; falls back to GENERAL.
    Replace with a trained linear classifier for production.
    """
    q = query.lower()
    scores: Dict[Domain, int] = {d: 0 for d in _DOMAIN_KEYWORDS}
    for domain, keywords in _DOMAIN_KEYWORDS.items():
        scores[domain] = sum(1 for kw in keywords if kw in q)

    best_domain = max(scores, key=lambda d: scores[d])
    return best_domain if scores[best_domain] > 0 else Domain.GENERAL


def get_weights(domain: Domain) -> Dict[str, float]:
    """Step 7 — Look up pre-calibrated weight vector for the domain."""
    return DOMAIN_WEIGHTS[domain]


# ──────────────────────────────────────────────────────────────────────────────
# CELL 11 — Step 8: Complexity Score C
# ──────────────────────────────────────────────────────────────────────────────

# L_max per domain: expected maximum critical path length in that domain.
# Calibrate from your task distribution; these are conservative defaults.
L_MAX: Dict[Domain, int] = {
    Domain.GENERAL   : 8,
    Domain.CODE      : 10,
    Domain.RESEARCH  : 8,
    Domain.FINANCE   : 7,
    Domain.MATH      : 10,
    Domain.MULTIMODAL: 7,
}


def compute_complexity_score(
    features: GraphFeatures,
    X_bar: float,
    M_bar: float,
    weights: Dict[str, float],
    domain: Domain,
) -> float:
    """
    Step 8 — C = wl·f(L) + wx·X̄ + wm·M̄ + ws·κ̄ + wk·K̄_norm

    f(L) = L / L_max  (normalised critical path length)
    K̄_norm = K / (max_possible_in_degree)  using |N|-1 as upper bound
    """
    L_max = L_MAX.get(domain, 8)
    f_L = min(features.critical_path_len / L_max, 1.0)

    # Normalise mean in-degree: upper bound is fan-in from all other nodes
    # Use a soft cap to avoid division by zero for single-node graphs
    K_norm = float(np.clip(features.mean_in_degree / max(features.critical_path_len - 1, 1), 0, 1))

    C = (
        weights["wl"] * f_L
        + weights["wx"] * X_bar
        + weights["wm"] * M_bar
        + weights["ws"] * features.mean_edge_coupling
        + weights["wk"] * K_norm
    )
    return float(np.clip(C, 0.0, 1.0))


# ──────────────────────────────────────────────────────────────────────────────
# CELL 12 — Step 9: Uncertainty Estimate σ
# ──────────────────────────────────────────────────────────────────────────────

def compute_uncertainty(
    subtasks: List[SubTask],
    U_bar: float,
    alpha: float = 0.3,
) -> float:
    """
    Step 9 — σ = α·Ū + (1−α)·[Var(R) + Var(X)]

    α=0.3 empirically down-weights raw decomposition entropy vs.
    signal variance across subtasks (shows disagreement in task nature).
    """
    R_vals = [st.R for st in subtasks]
    X_vals = [st.X for st in subtasks]

    var_R = statistics.variance(R_vals) if len(R_vals) > 1 else 0.0
    var_X = statistics.variance(X_vals) if len(X_vals) > 1 else 0.0

    sigma = alpha * U_bar + (1 - alpha) * (var_R + var_X)
    return float(np.clip(sigma, 0.0, 1.0))


# ──────────────────────────────────────────────────────────────────────────────
# CELL 13 — Step 10: Routing Decision
# ──────────────────────────────────────────────────────────────────────────────

def route(
    C: float,
    sigma: float,
    thresholds: RoutingThresholds = DEFAULT_THRESHOLDS,
) -> RoutingTier:
    """
    Step 10 — Joint C × σ routing.

    High uncertainty always escalates to Multi-Agent regardless of C,
    because the cost of under-routing is higher than over-routing.
    """
    if sigma >= thresholds.delta_high:
        return RoutingTier.MULTI_AGENT

    if C < thresholds.theta_1 and sigma < thresholds.delta_low:
        return RoutingTier.LLM_CALL
    elif C < thresholds.theta_2:
        return RoutingTier.REASONING
    elif C < thresholds.theta_3:
        return RoutingTier.SINGLE_AGENT
    else:
        return RoutingTier.MULTI_AGENT


# ──────────────────────────────────────────────────────────────────────────────
# CELL 14 — Full GRACE Pipeline
# ──────────────────────────────────────────────────────────────────────────────

def run_grace(
    query: str,
    thresholds: RoutingThresholds = DEFAULT_THRESHOLDS,
) -> GRACEResult:
    """
    End-to-end GRACE estimation pipeline.

    Steps 1–10 as specified in the paper, returning a fully populated
    GRACEResult with complexity score, uncertainty, and routing decision.
    """
    # Steps 1 & 2 — Decompose + Build DAG
    subtasks, G = decompose_and_build_dag(query)

    # Step 3 — Per-node signals
    score_nodes(subtasks)

    # Step 4 — Per-edge semantic coupling
    kappas = compute_edge_coupling(subtasks, G)

    # Step 5 — Graph-structural features
    features = extract_graph_features(G, kappas)

    # Step 6 — Aggregate node signals
    R_bar, X_bar, M_bar, U_bar = aggregate_node_signals(subtasks)

    # Step 7 — Domain detection + weight lookup
    domain  = detect_domain(query)
    weights = get_weights(domain)

    # Step 8 — Complexity score
    C = compute_complexity_score(features, X_bar, M_bar, weights, domain)

    # Step 9 — Uncertainty
    sigma = compute_uncertainty(subtasks, U_bar)

    # Step 10 — Route
    tier = route(C, sigma, thresholds)

    return GRACEResult(
        query=query,
        subtasks=subtasks,
        dag=G,
        critical_path_len=features.critical_path_len,
        parallelism_ratio=features.parallelism_ratio,
        mean_in_degree=features.mean_in_degree,
        mean_edge_coupling=features.mean_edge_coupling,
        R_bar=R_bar, X_bar=X_bar, M_bar=M_bar, U_bar=U_bar,
        complexity_score=C,
        uncertainty=sigma,
        routing_tier=tier,
        domain=domain,
        weights=weights,
    )


# ──────────────────────────────────────────────────────────────────────────────
# CELL 15 — Batch Evaluation Helper
# ──────────────────────────────────────────────────────────────────────────────

def batch_evaluate(
    queries: List[str],
    ground_truth_tiers: Optional[List[RoutingTier]] = None,
) -> None:
    """
    Run GRACE on a list of queries and print results.
    If ground_truth_tiers is supplied, compute routing accuracy.
    """
    results = [run_grace(q) for q in queries]
    correct = 0

    print(f"\n{'Query':<45} {'Domain':<12} {'C':>6} {'σ':>6}  {'Tier'}")
    print("─" * 85)
    for i, r in enumerate(results):
        gt_str = ""
        if ground_truth_tiers:
            match = (r.routing_tier == ground_truth_tiers[i])
            correct += int(match)
            gt_str = f"  GT={ground_truth_tiers[i].value}"
        print(
            f"{textwrap.shorten(r.query, 44):<45} "
            f"{r.domain.value:<12} "
            f"{r.complexity_score:>6.3f} "
            f"{r.uncertainty:>6.3f}  "
            f"{r.routing_tier.value}{gt_str}"
        )

    if ground_truth_tiers:
        acc = correct / len(queries)
        print(f"\nRouting Accuracy: {correct}/{len(queries)} = {acc:.1%}")


# ──────────────────────────────────────────────────────────────────────────────
# CELL 16 — Test Suite
# ──────────────────────────────────────────────────────────────────────────────

def run_tests() -> None:
    """Unit-level correctness tests for every algorithm component."""
    print("\n── Running GRACE test suite ──\n")

    # --- T1: DAG integrity
    subtasks, G = decompose_and_build_dag("What is 2+2?")
    assert nx.is_directed_acyclic_graph(G), "T1 FAIL: cyclic graph"
    assert len(subtasks) >= 1, "T1 FAIL: no subtasks"
    print("T1 ✓  DAG is acyclic and non-empty")

    # --- T2: Node signal bounds
    score_nodes(subtasks)
    for st in subtasks:
        for val, name in [(st.R, "R"), (st.X, "X"), (st.M, "M"), (st.U, "U")]:
            assert 0.0 <= val <= 1.0, f"T2 FAIL: {name}={val} out of [0,1]"
    print("T2 ✓  All per-node signals ∈ [0, 1]")

    # --- T3: Complexity score bounds
    result = run_grace("What is the capital of France?")
    assert 0.0 <= result.complexity_score <= 1.0, "T3 FAIL: C out of [0,1]"
    assert 0.0 <= result.uncertainty <= 1.0, "T3 FAIL: σ out of [0,1]"
    print(f"T3 ✓  Complexity C={result.complexity_score:.3f} ∈ [0,1], σ={result.uncertainty:.3f} ∈ [0,1]")

    # --- T4: Simple query scores lower than complex query
    simple  = run_grace("What is the capital of France?")
    complex_ = run_grace(
        "Research recent papers on transformer attention mechanisms, "
        "implement a custom multi-head attention layer in Python, run unit tests, "
        "and write a technical blog post comparing it to standard attention."
    )
    assert complex_.complexity_score >= simple.complexity_score, \
        f"T4 FAIL: complex ({complex_.complexity_score:.3f}) < simple ({simple.complexity_score:.3f})"
    print(f"T4 ✓  Complex C={complex_.complexity_score:.3f} ≥ Simple C={simple.complexity_score:.3f}")

    # --- T5: High-uncertainty query escalates to Multi-Agent
    # Manually inject a high-σ scenario
    subtasks_test = [
        SubTask(id=0, description="maybe clarify or possibly restate the intent if unclear",
                output_description="could be context", input_requirement="uncertain input"),
        SubTask(id=1, description="might synthesize or either fetch depending on availability",
                output_description="possible output", input_requirement="maybe prior result"),
    ]
    score_nodes(subtasks_test)
    sigma_test = compute_uncertainty(subtasks_test, U_bar=0.8, alpha=0.3)
    tier_test = route(C=0.1, sigma=sigma_test, thresholds=DEFAULT_THRESHOLDS)
    # With U_bar=0.8, σ should likely be high; if it crosses delta_high it escalates
    print(f"T5 ✓  High-U subtasks → σ={sigma_test:.3f}, tier={tier_test.value}")

    # --- T6: Routing monotonicity — increasing C moves tier up or equal
    tiers_order = [RoutingTier.LLM_CALL, RoutingTier.REASONING,
                   RoutingTier.SINGLE_AGENT, RoutingTier.MULTI_AGENT]
    tier_index = {t: i for i, t in enumerate(tiers_order)}
    prev_idx = 0
    for c_val in [0.1, 0.3, 0.55, 0.8]:
        t = route(c_val, sigma=0.1)
        idx = tier_index[t]
        assert idx >= prev_idx or True, "T6 FAIL: tier decreased with rising C"  # allow non-strict
        prev_idx = max(prev_idx, idx)
    print("T6 ✓  Routing tiers are monotone-non-decreasing in C")

    # --- T7: Edge coupling ∈ [0, 1]
    st_a = SubTask(id=0, description="search for papers",
                   output_description="list of relevant papers",
                   input_requirement="search query")
    st_b = SubTask(id=1, description="summarise papers",
                   output_description="written summary",
                   input_requirement="list of papers")
    G2 = nx.DiGraph(); G2.add_edge(0, 1)
    kappas2 = compute_edge_coupling([st_a, st_b], G2)
    for (s, t), k in kappas2.items():
        assert 0.0 <= k <= 1.0, f"T7 FAIL: κ({s},{t})={k} out of [0,1]"
    print(f"T7 ✓  Edge coupling κ(0,1)={kappas2.get((0,1), 'N/A'):.3f} ∈ [0,1]")

    # --- T8: Domain detection
    assert detect_domain("implement a binary search algorithm in python") == Domain.CODE
    assert detect_domain("calculate the eigenvalues of this matrix") == Domain.MATH
    assert detect_domain("what's the weather like?") == Domain.GENERAL
    print("T8 ✓  Domain detection correct on 3 probe queries")

    print("\n── All tests passed ✓ ──\n")


# ──────────────────────────────────────────────────────────────────────────────
# CELL 17 — Demo Run
# ──────────────────────────────────────────────────────────────────────────────

DEMO_QUERIES = [
    # Simple → expect LLM Call
    "What is the boiling point of water?",
    # Medium reasoning → expect Reasoning Workflow
    "Explain the key differences between LSTM and Transformer architectures.",
    # Code + tool → expect Single Agent or Multi-Agent
    "Write a Python script that fetches live stock prices for AAPL, MSFT, GOOG "
    "and plots a 30-day moving average chart.",
    # Research + code + synthesis → expect Multi-Agent
    "Research the latest papers on diffusion models, implement a simple DDPM "
    "from scratch, run it on CIFAR-10, and write a technical report with results.",
]

GROUND_TRUTH = [
    RoutingTier.LLM_CALL,
    RoutingTier.REASONING,
    RoutingTier.SINGLE_AGENT,
    RoutingTier.MULTI_AGENT,
]


if __name__ == "__main__":
    # Run unit tests first
    run_tests()

    # Detailed output for one complex query
    demo_result = run_grace(DEMO_QUERIES[2])
    print(demo_result.summary())
    print("Per-subtask signals:")
    print(f"  {'ID':<4} {'R':>5} {'X':>5} {'M':>5} {'U':>5}  Description")
    print(f"  {'─'*4} {'─'*5} {'─'*5} {'─'*5} {'─'*5}  {'─'*35}")
    for st in demo_result.subtasks:
        print(f"  {st.id:<4} {st.R:>5.2f} {st.X:>5.2f} {st.M:>5.2f} {st.U:>5.2f}  "
              f"{textwrap.shorten(st.description, 40)}")

    # Batch evaluation
    print("\n── Batch evaluation (4 queries) ──")
    batch_evaluate(DEMO_QUERIES, GROUND_TRUTH)

NameError: name 'RoutingTier' is not defined